In [1]:
import kagglehub
import shutil
import os

# Dataset herunterladen
download_path = kagglehub.dataset_download(
    "yasserh/instacart-online-grocery-basket-analysis-dataset"
)

# Zielordner festlegen
target_dir = "../data"

# Ordner erstellen (falls nicht vorhanden)
os.makedirs(target_dir, exist_ok=True)

# Alle Dateien rüberkopieren
for file in os.listdir(download_path):
    shutil.copy(
        os.path.join(download_path, file),
        os.path.join(target_dir, file)
    )

print("Alle Dateien wurden nach", target_dir, "kopiert")

c:\Users\jonat\Desktop\Repos\4. semester\PodSV Projekt\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Alle Dateien wurden nach ../data kopiert


In [ ]:
import os
import pickle
from collections import Counter
from itertools import combinations

import pandas as pd


def load_data(orders_path, order_products_path, products_path):
    orders = pd.read_csv(orders_path)
    order_products = pd.read_csv(order_products_path)
    products = pd.read_csv(products_path)
    data = order_products.merge(products, on="product_id")
    return orders, data


def build_ranking(data, orders):
    product_counts = Counter(data["product_name"])
    ranking = pd.DataFrame(product_counts.most_common(), columns=["Produkt", "Anzahl"])
    total_orders = orders["order_id"].nunique()
    ranking["Share in %"] = (ranking["Anzahl"] / total_orders * 100).round(2).astype(float)
    return ranking


def build_pairs(data):
    basket = data.groupby("order_id")["product_name"].apply(list)
    pairs = Counter()
    for items in basket:
        for pair in combinations(sorted(set(items)), 2):
            pairs[pair] += 1
    return pairs


def save_processed(data, ranking, pairs, out_dir="../data"):
    os.makedirs(out_dir, exist_ok=True)
    with open(f"{out_dir}/data.pkl", "wb") as f:
        pickle.dump(data, f)
    with open(f"{out_dir}/ranking.pkl", "wb") as f:
        pickle.dump(ranking, f)
    with open(f"{out_dir}/pairs.pkl", "wb") as f:
        pickle.dump(pairs, f)
    print(f"Gespeichert in {out_dir}/")


orders, data = load_data(
    "../data/orders.csv",
    "../data/order_products__prior.csv",
    "../data/products.csv"
)

ranking = build_ranking(data, orders)
pairs   = build_pairs(data)
save_processed(data, ranking, pairs)